# Day 18: Scope, Default Values and Functions Calling Functions

**Today's question:** which variables can a function "see", and how do we build something big out of small
functions?

Run each cell with **Shift+Enter**. Before running a cell marked 🤔, predict what it will show.

## 1. Variables made inside a function are **local**

In [ ]:
def make_greeting(name):
    """Return a greeting for name."""
    message = f"Hi {name}!"
    return message


print(make_greeting("Zara"))

In [ ]:
print(message)

Every call gets its own **fresh whiteboard** (its *local scope*). The parameters and any variables created in the
body are written on it. When the function returns, the whiteboard is **wiped**. Only the returned value survives.

```
GLOBAL (the notebook / the file)        while make_greeting("Zara") runs
┌──────────────────────────────┐        ┌───────────────────────────┐
│ make_greeting → (function)   │        │ name    → "Zara"          │
│                              │        │ message → "Hi Zara!"      │
└──────────────────────────────┘        └───────────────────────────┘
                                          wiped when the call returns
```

To get something **out** of a function, `return` it. That is the only door out.

## 2. Same name, different variables

In [ ]:
total = 0


def add_up(prices):
    """Return the sum of a list of prices."""
    total = 0
    for price in prices:
        total += price
    return total

🤔 **Predict** both lines:

In [ ]:
print(add_up([3, 4, 5]))
print(total)

There are **two** variables called `total`: one on the global board (`0`) and one on the function's whiteboard
(`12`). The function only changed its own.

```
GLOBAL                  add_up([3, 4, 5])
┌─────────────┐         ┌────────────────────────┐
│ total → 0   │         │ prices → [3, 4, 5]     │
└─────────────┘         │ total  → 0, 3, 7, 12   │
                        └────────────────────────┘
```

Rule: a function looks for a name **on its own whiteboard first**, then on the global board.

## 3. Reading a global variable: works, but is risky

Reading a global *constant* (a value that never changes, written in `UPPER_CASE`) is fine:

In [ ]:
SERVICE_RATE = 0.10  # constant: never reassigned


def add_service(bill):
    """Return the bill plus the service charge."""
    return bill + bill * SERVICE_RATE


print(add_service(40))

But a function that reads a global variable which **changes** has a hidden input. 🤔 **Predict:**

In [ ]:
discount = 20


def sale_price(price):
    """Return price minus the current discount."""
    return price - discount


print(sale_price(100))
discount = 50
print(sale_price(100))

Same call, different answers, and nothing in the call `sale_price(100)` tells you why. The fix: make the hidden
input a **parameter**.

In [ ]:
def sale_price(price, discount):
    """Return price minus discount."""
    return price - discount


print(sale_price(100, 20))
print(sale_price(100, 50))

## 4. Changing a global from inside a function

In [ ]:
visitors = 0


def add_visitor():
    """Count one more visitor."""
    visitors = visitors + 1

In [ ]:
add_visitor()

Because `visitors = ...` **assigns** inside the function, Python treats `visitors` as a *local* name for the
whole function. Then `visitors + 1` tries to read the local one before it has a value.

Python has a keyword that forces a function to change a global (you will see it in other people's code):

```python
def add_visitor():
    global visitors          # works, but we avoid it in this course
    visitors = visitors + 1
```

We avoid `global` because any function could change the value at any time: the bug appears far away from its
cause, and you cannot check the function on its own. **Pass values in, return them out:**

In [ ]:
def add_visitor(count):
    """Return the visitor count plus one."""
    return count + 1


visitors = 0
visitors = add_visitor(visitors)
visitors = add_visitor(visitors)
print(visitors)

## 5. Default parameter values

A parameter can have a **default value**, used when the caller leaves that argument out.

In [ ]:
def add_tax(amount, rate=0.10):
    """Return amount plus tax at rate (10% unless told otherwise)."""
    return amount + amount * rate


print(add_tax(200))
print(add_tax(200, 0.05))
print(add_tax(200, rate=0))

You have been using defaults all along: `round` rounds to 0 decimals unless told otherwise, and `print`
separates values with a space unless you give it `sep=`.

In [ ]:
print(round(3.14159), round(3.14159, 2))
print("Tea", "Coffee", "Juice")
print("Tea", "Coffee", "Juice", sep=" | ")

🤔 **Predict** the three name tags:

In [ ]:
def name_tag(name, role="Student", event="Python Club"):
    """Return the text for a name tag."""
    return f"{name} | {role} | {event}"


print(name_tag("Ken"))
print(name_tag("Mei", "Mentor"))
print(name_tag("Omar", event="Hack Day"))

Use a **keyword argument** to skip over a default you want to keep (`event=` above keeps `role="Student"`).

## 6. Functions calling functions: a price calculator

Small functions, each with **one job**, can be combined into a bigger one. This is called **composition**.

In [ ]:
def subtotal(prices):
    """Return the sum of a list of prices."""
    return sum(prices)


def apply_discount(amount, percent=0):
    """Return amount after a percent discount (no discount by default)."""
    return amount - amount * percent / 100


def format_money(amount):
    """Return amount as text with a thousands comma and 2 decimals."""
    return f"{amount:,.2f}"

In [ ]:
def checkout_total(prices, discount_percent=0):
    """Return the amount to pay for a list of prices, formatted as text."""
    amount = subtotal(prices)
    amount = apply_discount(amount, discount_percent)
    amount = add_tax(amount)
    return format_money(amount)

🤔 **Predict** both totals (tax is 10% by default):

In [ ]:
basket = [450, 1200, 350]
print(checkout_total(basket))
print(checkout_total(basket, discount_percent=10))

What happens during the second call. Each small function returns to the one that called it:

```
checkout_total([450, 1200, 350], 10)
 ├─ subtotal([450, 1200, 350])     → 2000
 ├─ apply_discount(2000, 10)       → 1800.0
 ├─ add_tax(1800.0)                → 1980.0
 └─ format_money(1980.0)           → "1,980.00"
```

`checkout_total` reads like the recipe. Each step can be changed or checked on its own. The "return, don't
print" rule from Day 17 is what makes this possible.

> **Optional box: `*args`.** You may see `def total(*prices):` in other code. The `*` means "collect any number
> of arguments into a tuple". We don't use it in this course: a list parameter does the same job more clearly.

## 7. Common mistakes

Mistake 1 (`NameError` for a local variable used outside) is in section 1. Mistake 2 (`UnboundLocalError` when
changing a global) is in section 4. These cells fail **on purpose**:

In [ ]:
# Mistake 3: a parameter without a default after one with a default
def name_tag(role="Student", name):
    return f"{name} | {role}"

**Mistake 4: a middle function forgets to `return`.** The error appears in a *later* function, but the bug is in
the earlier one.

In [ ]:
def apply_discount(amount, percent=0):
    """Return amount after a percent discount (no discount by default)."""
    amount - amount * percent / 100  # oops: no return

In [ ]:
checkout_total(basket, discount_percent=10)

The error message points at `add_tax`, but `add_tax` is fine: it received `None`. Read the whole message
(the arrows show the chain of calls) and ask: **where did the `None` come from?**

In [ ]:
def apply_discount(amount, percent=0):
    """Return amount after a percent discount (no discount by default)."""
    return amount - amount * percent / 100


print(checkout_total(basket, discount_percent=10))

## Recap

1. Variables created inside a function (and its parameters) are **local**: they disappear when it returns.
2. A function looks on its own whiteboard first, then on the global board. Reading global **constants** is fine;
   relying on changing globals is not. Pass values in, return them out. Avoid `global`.
3. `def f(a, b=10)`: `b` has a **default**. Parameters with defaults come last.
4. Small one-job functions that **return** can be combined into bigger ones (**composition**).

**Next time:** how to plan a whole program as functions *before* writing it, and how to check each function.